# 04 冻结退出后的持仓时间审计

本 Notebook 读取 03 号 Notebook 生成的五列 CSV，比较原始三状态与加入两侧冻结退出后的最终三状态：

- 按实际 `date` 统计各状态连续持有交易日数；
- 比较总天数、段数、均值、中位数和最长段；
- 列出每一个具体退出日；
- 展示每个原始非零状态段被中性化了多少天。

最终状态采用事件日退出口径：只有实际触发 `-1→0` 或 `1→0` 的当天置为 0；下一交易日重新跟随五列 CSV 中的基础三状态，不会把退出日之后的同侧日期全部压成 0。

Notebook 最后一节提供可直接截图的全周期/分阶段准确性汇总和最终价格曲线图：点形状区分两侧，绿色/红色区分退出后 O2O 有利/不利，点旁数字为 H1 O2O(bp)。

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', message=r'Glyph .* missing from current font')

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 60)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "pool_runner.py").is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
CSV_PATH = Path(
    os.environ.get(
        "1545_SIGNAL_CSV_PATH",
        str(PACKAGE_ROOT / "IC_1545_frozen_exit_signals.csv"),
    )
).expanduser()

required = [
    "date",
    "three_state",
    "minus_exit_signal",
    "plus_exit_signal",
    "final_three_state",
]
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"找不到 03 号 Notebook 生成的 CSV: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少列: {missing}; 实际列={list(df.columns)}")

df = df[required].copy()
df["date"] = pd.to_datetime(df["date"], errors="raise").dt.normalize()
df = df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
for column in ["three_state", "minus_exit_signal", "plus_exit_signal", "final_three_state"]:
    df[column] = pd.to_numeric(df[column], errors="raise").astype(int)

assert df["three_state"].isin([-1, 0, 1]).all()
assert df["final_three_state"].isin([-1, 0, 1]).all()
assert df["minus_exit_signal"].isin([0, 1]).all()
assert df["plus_exit_signal"].isin([0, 1]).all()

print("读取文件:", CSV_PATH.resolve())
print("行数:", len(df), "日期:", df["date"].min().date(), "->", df["date"].max().date())
display(df.head(10))

## 1. 原始状态与最终状态的总体变化

In [ ]:
def state_counts(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    return pd.DataFrame({
        "state": [-1, 0, 1],
        "days": [int((frame[column] == value).sum()) for value in (-1, 0, 1)],
    })

count_table = state_counts(df, "three_state").rename(columns={"days": "original_days"})
count_table["final_days"] = state_counts(df, "final_three_state")["days"]
count_table["delta_days_final_minus_original"] = (
    count_table["final_days"] - count_table["original_days"]
)
count_table["meaning"] = count_table["state"].map({
    -1: "负向持仓",
    0: "中性",
    1: "正向持仓",
})
display(count_table[[
    "state", "meaning", "original_days", "final_days",
    "delta_days_final_minus_original",
]])

print("原始状态分布：")
display(df["three_state"].value_counts().sort_index().rename("days").to_frame())
print("最终状态分布：")
display(df["final_three_state"].value_counts().sort_index().rename("days").to_frame())

# 用实际执行日统计占比；三列占比各自加总为 100%。
share_table = count_table[["state", "meaning", "original_days", "final_days"]].copy()
share_table["original_share_pct"] = share_table["original_days"] / len(df) * 100
share_table["final_share_pct"] = share_table["final_days"] / len(df) * 100
share_table["share_delta_pct_point"] = share_table["final_share_pct"] - share_table["original_share_pct"]
print("状态占比与时间变化（实际交易日）：")
display(share_table.round(3))

# 同时按 Development/Validation/Test（按实际 date）拆分，便于截图观察三个时期是否结构一致。
df["period"] = np.select(
    [df["date"] <= pd.Timestamp("2022-12-31"), df["date"] <= pd.Timestamp("2024-12-31")],
    ["Development", "Validation"],
    default="Test",
)
period_rows = []
for period, group in df.groupby("period", sort=False):
    for state, label in [(-1, "负向持仓"), (0, "中性"), (1, "正向持仓")]:
        original_days = int(group["three_state"].eq(state).sum())
        final_days = int(group["final_three_state"].eq(state).sum())
        period_rows.append({
            "period": period, "state": state, "meaning": label,
            "period_days": int(len(group)),
            "original_days": original_days, "original_share_pct": original_days / len(group) * 100,
            "final_days": final_days, "final_share_pct": final_days / len(group) * 100,
            "share_delta_pct_point": (final_days - original_days) / len(group) * 100,
        })
period_share_table = pd.DataFrame(period_rows)
display(period_share_table.round(3))

## 2. 各状态连续持仓段：段数、均值、中位数和最长段

In [ ]:
def run_table(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    work = frame[["date", column]].copy()
    work["run_id"] = work[column].ne(work[column].shift()).cumsum()
    rows = []
    for run_id, group in work.groupby("run_id", sort=True):
        rows.append({
            "run_id": int(run_id),
            "state": int(group[column].iloc[0]),
            "start_date": group["date"].iloc[0].date().isoformat(),
            "end_date": group["date"].iloc[-1].date().isoformat(),
            "trading_days": int(len(group)),
        })
    return pd.DataFrame(rows)

original_runs = run_table(df, "three_state")
final_runs = run_table(df, "final_three_state")

def duration_summary(runs: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    for state in (-1, 0, 1):
        values = runs.loc[runs["state"].eq(state), "trading_days"]
        rows.append({
            "series": label,
            "state": state,
            "segments": int(len(values)),
            "total_days": int(values.sum()),
            "mean_days": float(values.mean()) if len(values) else np.nan,
            "median_days": float(values.median()) if len(values) else np.nan,
            "max_days": int(values.max()) if len(values) else 0,
            "min_days": int(values.min()) if len(values) else 0,
        })
    return pd.DataFrame(rows)

duration_table = pd.concat([
    duration_summary(original_runs, "original_three_state"),
    duration_summary(final_runs, "final_three_state"),
], ignore_index=True)
display(duration_table.round(3))

print("原始三状态持仓段（非零）：")
display(original_runs.loc[original_runs["state"].ne(0)].reset_index(drop=True))
print("最终三状态持仓段（非零）：")
display(final_runs.loc[final_runs["state"].ne(0)].reset_index(drop=True))

## 3. 每个具体退出日

In [ ]:
exit_rows = df.loc[
    df["minus_exit_signal"].eq(1) |
    df["plus_exit_signal"].eq(1)
].copy()
if exit_rows.empty:
    print("没有退出日。")
else:
    exit_rows.insert(
        1,
        "exit_side",
        np.select(
            [
                exit_rows["minus_exit_signal"].eq(1),
                exit_rows["plus_exit_signal"].eq(1),
            ],
            ["-1→0", "1→0"],
            default="both",
        ),
    )
    exit_rows.insert(3, "state_after_exit", exit_rows["final_three_state"])
    print("退出日明细（date 是实际执行开盘日）：")
    display(exit_rows)

    print("退出日按年份统计：")
    exit_year = exit_rows.assign(year=exit_rows["date"].dt.year)
    display(
        exit_year.groupby(["year", "exit_side"], as_index=False)
        .agg(exit_days=("date", "size"))
        .sort_values(["year", "exit_side"])
    )

## 4. 每个原始非零状态段被缩短了多少

In [ ]:
df["original_run_id"] = df["three_state"].ne(df["three_state"].shift()).cumsum()
segment_rows = []
for run_id, group in df.loc[df["three_state"].ne(0)].groupby("original_run_id", sort=True):
    original_state = int(group["three_state"].iloc[0])
    neutralized = group["final_three_state"].eq(0)
    segment_rows.append({
        "original_run_id": int(run_id),
        "original_state": original_state,
        "start_date": group["date"].iloc[0].date().isoformat(),
        "end_date": group["date"].iloc[-1].date().isoformat(),
        "original_days": int(len(group)),
        "final_same_side_days": int(group["final_three_state"].eq(original_state).sum()),
        "neutralized_days": int(neutralized.sum()),
        "first_exit_date": (
            group.loc[neutralized, "date"].iloc[0].date().isoformat()
            if neutralized.any() else ""
        ),
        "exit_days": ", ".join(
            group.loc[
                (
                    group["minus_exit_signal"].eq(1) |
                    group["plus_exit_signal"].eq(1)
                ),
                "date",
            ].dt.strftime("%Y-%m-%d").tolist()
        ),
    })

segment_audit = pd.DataFrame(segment_rows)
if segment_audit.empty:
    print("没有非零原始状态段。")
else:
    print("发生中性化的原始非零段：")
    display(segment_audit.loc[segment_audit["neutralized_days"].gt(0)])
    print("全部原始非零段审计：")
    display(segment_audit)

# 这一摘要专门回答：退出是否制造了大量碎片化持仓，以及退出后中性通常持续多久。
fragment_rows = []
for state, label in [(-1, "负向"), (1, "正向")]:
    original_values = original_runs.loc[original_runs["state"].eq(state), "trading_days"]
    final_values = final_runs.loc[final_runs["state"].eq(state), "trading_days"]
    affected = segment_audit.loc[
        (segment_audit["original_state"].eq(state)) &
        (segment_audit["neutralized_days"].gt(0))
    ]
    fragment_rows.append({
        "side": label,
        "original_segments": int(len(original_values)),
        "final_segments": int(len(final_values)),
        "final_segments_<=3d": int((final_values <= 3).sum()),
        "final_segments_<=5d": int((final_values <= 5).sum()),
        "original_median_days": float(original_values.median()),
        "final_median_days": float(final_values.median()),
        "exit_affected_segments": int(len(affected)),
        "exit_neutralized_days_median": float(affected["neutralized_days"].median()) if len(affected) else np.nan,
        "exit_neutralized_days_min": int(affected["neutralized_days"].min()) if len(affected) else 0,
        "exit_neutralized_days_max": int(affected["neutralized_days"].max()) if len(affected) else 0,
    })

fragment_summary = pd.DataFrame(fragment_rows)
print("碎片化与退出后中性化摘要（天数均为实际交易日）：")
display(fragment_summary.round(3))
print("解释：事件日置 0 会把原始同侧段在信号日切开，因此 final_segments 可能多于 original_segments；这不是后续日期持续压制，而是逐个实际信号日产生的中性插口。请同时结合 final_segments_<=3d 和退出日期明细判断碎片化程度。")

# 退出标记所在的最终 0 段还包含原本就为空仓的日期，单独列出以免混淆。
final_run_work = df[["date", "final_three_state", "minus_exit_signal", "plus_exit_signal"]].copy()
final_run_work["final_run_id"] = final_run_work["final_three_state"].ne(final_run_work["final_three_state"].shift()).cumsum()
final_zero_runs = final_run_work.groupby("final_run_id", as_index=False).agg(
    final_state=("final_three_state", "first"),
    start_date=("date", "min"),
    end_date=("date", "max"),
    trading_days=("date", "size"),
    minus_exit_days=("minus_exit_signal", "sum"),
    plus_exit_days=("plus_exit_signal", "sum"),
)
exit_zero_runs = final_zero_runs.loc[
    (final_zero_runs["final_state"].eq(0)) &
    (final_zero_runs[["minus_exit_days", "plus_exit_days"]].sum(axis=1).gt(0))
].copy()
print("包含退出标记的最终中性段（包含后续普通中性期）：")
display(exit_zero_runs)
print("这张表的 trading_days 会把退出后的普通中性期也算进去；前一张摘要的 exit_neutralized_days_* 才是退出发生时仍处于原始同侧段内的持续天数。")

print("HOLDING_AUDIT_END")

## 5. 状态与指数价格曲线、退出日图

以下三张图都按五列 CSV 的 `date`（形成日后的实际执行日）对齐现货指数收盘价：

1. 原始 `three_state`；2. 退出调整后的 `final_three_state`；3. 退出事件本身。

远端运行时默认从唯一现货输入读取价格；本地如需指定价格文件，可设置 `1545_PRICE_PATH`。普通价格文件需含日期、`open`、`close`；若使用本地缓存面板，则会自动采用 `effective_date/close_t1/open_t1/open_t2`，并按实际执行日计算 O2O H1。

In [ ]:
def _read_price_curve() -> tuple[pd.DataFrame | None, str | None]:
    configured = os.environ.get("1545_PRICE_PATH", "").strip()
    try:
        if configured:
            price_path = Path(configured).expanduser()
            if not price_path.is_file():
                raise FileNotFoundError(price_path)
            raw = pd.read_parquet(price_path) if price_path.suffix.lower() == ".parquet" else pd.read_csv(price_path, encoding="utf-8-sig")
        else:
            from common.data import _read_spot, load_paths
            _, paths = load_paths()
            price_path = paths["spot"]
            raw = _read_spot(price_path).reset_index()
        if isinstance(raw.index, pd.DatetimeIndex):
            raw = raw.reset_index()

        # 本地 canonical_panel 使用 formation_date 索引，但 CSV 的 date 是 effective_date。
        # 优先使用缓存面板已经对齐好的 effective_date/close_t1/open_t1/open_t2。
        if {"effective_date", "close_t1", "open_t1", "open_t2"}.issubset(raw.columns):
            curve = pd.DataFrame({
                "date": pd.to_datetime(raw["effective_date"], errors="coerce").dt.normalize(),
                "close": pd.to_numeric(raw["close_t1"], errors="coerce"),
                "open_current": pd.to_numeric(raw["open_t1"], errors="coerce"),
                "open_next": pd.to_numeric(raw["open_t2"], errors="coerce"),
            })
        else:
            date_candidates = ["date", "trade_dt", "trade_date", "formation_date"]
            date_col = next((name for name in date_candidates if name in raw.columns), None)
            close_col = next((name for name in raw.columns if str(name).lower() == "close"), None)
            open_col = next((name for name in raw.columns if str(name).lower() == "open"), None)
            if date_col is None or close_col is None:
                raise ValueError(f"价格文件必须含日期列和 close 列；实际列={list(raw.columns)}")
            curve = pd.DataFrame({
                "date": pd.to_datetime(raw[date_col], errors="coerce").dt.normalize(),
                "close": pd.to_numeric(raw[close_col], errors="coerce"),
                "open_current": pd.to_numeric(raw[open_col], errors="coerce") if open_col else np.nan,
            })
            curve = curve.sort_values("date").drop_duplicates("date", keep="last")
            curve["open_next"] = curve["open_current"].shift(-1)
        curve["o2o_h1_bp"] = curve["open_next"].div(curve["open_current"]).sub(1).mul(10000)
        curve = curve.dropna(subset=["date", "close"]).sort_values("date").drop_duplicates("date", keep="last")
        return curve, str(price_path)
    except Exception as exc:
        print("PRICE_PLOT_SKIPPED:", repr(exc))
        print("远端默认会读取现货路径；本地可设置 1545_PRICE_PATH 后重新执行本节。")
        return None, None

price_curve, price_source = _read_price_curve()
if price_curve is not None:
    plot_df = df.merge(price_curve, on="date", how="left").dropna(subset=["close"]).copy()
    print("价格来源:", price_source)
    print("可绘制共同日期:", len(plot_df), "/", len(df), "；价格范围:", plot_df["date"].min().date(), "->", plot_df["date"].max().date())

    state_colors = {-1: "#d62728", 0: "#7f7f7f", 1: "#2ca02c"}
    state_labels = {-1: "-1 负向", 0: "0 中性", 1: "+1 正向"}

    def plot_state_on_price(column: str, title: str) -> None:
        fig, ax = plt.subplots(figsize=(18, 6))
        ax.plot(plot_df["date"], plot_df["close"], color="#444444", linewidth=1.0, label="CSI500 close", zorder=1)
        for state in (-1, 0, 1):
            part = plot_df.loc[plot_df[column].eq(state)]
            ax.scatter(part["date"], part["close"], s=12, alpha=0.78, color=state_colors[state], label=state_labels[state], zorder=2)
        ax.set_title(title)
        ax.set_xlabel("实际执行日 date")
        ax.set_ylabel("指数收盘价")
        ax.grid(alpha=0.22)
        ax.legend(ncol=4, loc="upper left")
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

    plot_state_on_price("three_state", "原始三状态与 CSI500 指数价格")
    plot_state_on_price("final_three_state", "退出调整后三状态与 CSI500 指数价格")

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.plot(plot_df["date"], plot_df["close"], color="#444444", linewidth=1.0, label="CSI500 close", zorder=1)
    minus = plot_df.loc[plot_df["minus_exit_signal"].eq(1)].copy()
    plus = plot_df.loc[plot_df["plus_exit_signal"].eq(1)].copy()
    # 三角号缩小，避免长周期图上遮挡价格曲线；O2O 数字同时写入图和下方明细表。
    ax.scatter(minus["date"], minus["close"], s=18, alpha=0.9, color="#d62728", marker="v", label="下跌退出（-1→0）", zorder=3)
    ax.scatter(plus["date"], plus["close"], s=18, alpha=0.9, color="#1f77b4", marker="^", label="上涨退出（1→0）", zorder=3)
    for part, color, offset in [(minus, "#d62728", -7), (plus, "#1f77b4", 5)]:
        for _, row in part.dropna(subset=["o2o_h1_bp"]).iterrows():
            ax.annotate(
                f"{row['o2o_h1_bp']:+.1f}bp",
                xy=(row["date"], row["close"]),
                xytext=(0, offset), textcoords="offset points",
                fontsize=5.5, color=color, rotation=90, ha="center", va="center", alpha=0.82,
            )
    ax.set_title("退出发生日与 CSI500 指数价格（标注退出后 O2O H1）")
    ax.set_xlabel("实际执行日 date")
    ax.set_ylabel("指数收盘价")
    ax.grid(alpha=0.22)
    ax.legend(loc="upper left")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    exit_o2o = pd.concat([
        minus.assign(exit_side="下跌退出（-1→0）"),
        plus.assign(exit_side="上涨退出（1→0）"),
    ], ignore_index=True).sort_values("date")
    # raw_o2o 是指数本身的原始开盘到开盘收益；aligned 是按退出方向折算后的收益。
    exit_o2o["exit_aligned_h1_bp"] = np.where(
        exit_o2o["exit_side"].str.startswith("下跌"),
        exit_o2o["o2o_h1_bp"],
        -exit_o2o["o2o_h1_bp"],
    )
    exit_o2o_table = exit_o2o[["date", "exit_side", "close", "o2o_h1_bp", "exit_aligned_h1_bp"]].copy()
    print("退出日对应的 O2O H1：o2o_h1_bp 是原始指数收益；exit_aligned_h1_bp 是按退出方向折算后的收益（单位 bp）：")
    print("图中标注的是原始 o2o_h1_bp。")
    display(exit_o2o_table.round({"close": 3, "o2o_h1_bp": 2}))
    print("退出日 O2O 汇总：")
    display(
        exit_o2o.groupby("exit_side", as_index=False).agg(
            n=("o2o_h1_bp", "count"),
            mean_o2o_h1_bp=("o2o_h1_bp", "mean"),
            median_o2o_h1_bp=("o2o_h1_bp", "median"),
            raw_win_rate=("o2o_h1_bp", lambda x: (x > 0).mean()),
            mean_exit_aligned_h1_bp=("exit_aligned_h1_bp", "mean"),
            median_exit_aligned_h1_bp=("exit_aligned_h1_bp", "median"),
            aligned_win_rate=("exit_aligned_h1_bp", lambda x: (x > 0).mean()),
            min_o2o_h1_bp=("o2o_h1_bp", "min"),
            max_o2o_h1_bp=("o2o_h1_bp", "max"),
        ).round(3)
    )
    print("下跌退出点数:", len(minus), "；上涨退出点数:", len(plus))
else:
    print("没有价格曲线，三张图未绘制。")

## 6. 冻结后持仓段与收益分布审计

这一节是冻结完成后的描述性审计，只用于确认退出调整后的三状态质量，不回写候选池、版本入选或 Top1 冻结规则。除持仓段结构外，还分别给出负向、零、正向的段收益和日收益分布，并检查是否出现大量一天持仓。收益采用现货实际执行日开盘到开盘（O2O）口径；如果没有价格文件，则仍会输出结构统计，收益列显示为缺失。

另外专门输出退出质量标志：段胜率是否提高、方向对齐正收益段数量是否减少、3%/5%大额正收益尾部是否缩短，以及负收益尾部是否改善。这些标志只用于观察，不参与筛选。对于负向 `-1`，还会单独输出原始指数段收益分布，重点观察原来上涨幅度较大的不利段是否被退出切短。


In [ ]:
from state_segment_audit import audit_state_adjustment

# 复用本节前面按 1545_PRICE_PATH 读取的现货价格，避免审计再次回退到远端路径。
audit_price = None
audit_price_source = None
if price_curve is not None:
    audit_price = (
        price_curve[["date", "open_current"]]
        .rename(columns={"open_current": "open"})
        .set_index("date")
    )
    audit_price_source = price_source

segment_audit = audit_state_adjustment(
    df,
    price=audit_price,
    price_source=audit_price_source,
)
print("价格来源:", segment_audit["price_source"])
print("信号日期:", segment_audit["signal_date_min"], "至", segment_audit["signal_date_max"])
print("收益可用日数:", segment_audit["return_available_daily_rows"],
      "；收益缺失日数:", segment_audit["return_missing_daily_rows"])
print("selection_used =", segment_audit["selection_used"],
      "（本节仅观察，不参与候选/版本筛选）")

print("\n原始三状态 vs 退出调整后三状态：")
display(segment_audit["comparison"].round(4))

print("\n持仓段统计：段数、持仓天数、平均/中位/最长段、一天段与三天内碎片段占比、段胜率：")
display(segment_audit["segment_summary"].round(4))

print("\n持仓段 O2O 方向收益分布（百分比）：")
display(segment_audit["segment_distribution"].round(4))
print("\n原始指数持仓段收益分布（百分比；不做方向对齐，负向上涨为不利尾部）：")
display(segment_audit["raw_segment_distribution"].round(4))
print("\n原始指数不利尾部紧凑对照（负向上涨段；仅观察）：")
display(segment_audit["raw_tail_comparison"].round(4))

print("\n逐交易日 O2O 方向收益分布（百分比）：")
display(segment_audit["daily_distribution"].round(4))
print("\n原始指数逐交易日 O2O 收益分布（百分比）：")
display(segment_audit["raw_daily_distribution"].round(4))

print("\n退出质量标志（仅观察，不参与筛选）：")
display(segment_audit["quality_flags"])


## 7. 最终截图汇总：退出预测准确性与 O2O

本节放在 Notebook 最后，专门提供一页式可截图结果。绿色表示退出后 O2O 对原方向有利（预测准确），红色表示不利；图中数字为退出执行日之后的 H1 O2O 原始指数收益，单位 bp。负向退出的有利情形是指数上涨，正向退出的有利情形是指数下跌。

准确性只作为冻结后的描述性观察，不参与候选池、版本入选或 Top1 冻结。

In [ ]:
print('FINAL_SCREENSHOT_SUMMARY')
if price_curve is None or 'exit_o2o' not in globals():
    print('没有可用价格曲线，无法计算退出后的 O2O 准确性。')
else:
    # 每一行对应一个实际退出信号日；颜色按退出后 H1 O2O 是否有利定义。
    accuracy_events = exit_o2o.copy()
    accuracy_events = accuracy_events.drop(columns=['period'], errors='ignore').merge(
        df[['date', 'period']], on='date', how='left'
    )
    accuracy_events['o2o_h1_bp'] = pd.to_numeric(accuracy_events['o2o_h1_bp'], errors='coerce')
    accuracy_events['exit_aligned_h1_bp'] = pd.to_numeric(
        accuracy_events['exit_aligned_h1_bp'], errors='coerce'
    )
    accuracy_events['prediction_correct'] = accuracy_events['exit_aligned_h1_bp'].gt(0)
    accuracy_events.loc[accuracy_events['exit_aligned_h1_bp'].isna(), 'prediction_correct'] = pd.NA
    accuracy_events['prediction_result'] = np.select(
        [accuracy_events['exit_aligned_h1_bp'].gt(0), accuracy_events['exit_aligned_h1_bp'].lt(0)],
        ['有利/准确', '不利/错误'],
        default='收益缺失',
    )

    side_order = ['下跌退出（-1→0）', '上涨退出（1→0）']
    side_rows = []
    for side in side_order:
        part = accuracy_events.loc[accuracy_events['exit_side'].eq(side)].copy()
        valid = part.dropna(subset=['exit_aligned_h1_bp'])
        aligned = valid['exit_aligned_h1_bp']
        raw = valid['o2o_h1_bp']
        side_rows.append({
            'exit_side': side,
            'signal_days': int(len(part)),
            'o2o_available_days': int(len(valid)),
            'favorable_days': int(aligned.gt(0).sum()),
            'unfavorable_days': int(aligned.lt(0).sum()),
            'accuracy_pct': float(aligned.gt(0).mean() * 100) if len(valid) else np.nan,
            'raw_o2o_mean_bp': float(raw.mean()) if len(valid) else np.nan,
            'raw_o2o_median_bp': float(raw.median()) if len(valid) else np.nan,
            'raw_o2o_p25_bp': float(raw.quantile(.25)) if len(valid) else np.nan,
            'raw_o2o_p75_bp': float(raw.quantile(.75)) if len(valid) else np.nan,
            'aligned_o2o_mean_bp': float(aligned.mean()) if len(valid) else np.nan,
            'aligned_o2o_median_bp': float(aligned.median()) if len(valid) else np.nan,
            'aligned_o2o_p25_bp': float(aligned.quantile(.25)) if len(valid) else np.nan,
            'aligned_o2o_p75_bp': float(aligned.quantile(.75)) if len(valid) else np.nan,
        })
    accuracy_summary = pd.DataFrame(side_rows)
    print('全周期退出预测准确性（H1 O2O）：')
    display(accuracy_summary.round(2))

    period_rows = []
    for period in ['Development', 'Validation', 'Test']:
        for side in side_order:
            part = accuracy_events.loc[
                accuracy_events['period'].eq(period) & accuracy_events['exit_side'].eq(side)
            ]
            valid = part.dropna(subset=['exit_aligned_h1_bp'])
            aligned = valid['exit_aligned_h1_bp']
            period_rows.append({
                'period': period,
                'exit_side': side,
                'signal_days': int(len(part)),
                'favorable_days': int(aligned.gt(0).sum()),
                'accuracy_pct': float(aligned.gt(0).mean() * 100) if len(valid) else np.nan,
                'raw_o2o_mean_bp': float(valid['o2o_h1_bp'].mean()) if len(valid) else np.nan,
                'aligned_o2o_mean_bp': float(aligned.mean()) if len(valid) else np.nan,
                'aligned_o2o_median_bp': float(aligned.median()) if len(valid) else np.nan,
            })
    period_accuracy_summary = pd.DataFrame(period_rows)
    print('分阶段退出预测准确性（Test 只在冻结后观察）：')
    display(period_accuracy_summary.round(2))

    # 一页式核心质量仪表盘：同时看信号准确性、段胜率、收益和碎片化。
    segment_summary_for_dashboard = segment_audit['segment_summary']
    dashboard_rows = []
    for state, side in [(-1, '下跌退出（-1→0）'), (1, '上涨退出（1→0）')]:
        original = segment_summary_for_dashboard.loc[
            (segment_summary_for_dashboard['series'].eq('original')) &
            (segment_summary_for_dashboard['state'].eq(state))
        ].iloc[0]
        adjusted = segment_summary_for_dashboard.loc[
            (segment_summary_for_dashboard['series'].eq('adjusted')) &
            (segment_summary_for_dashboard['state'].eq(state))
        ].iloc[0]
        event_part = accuracy_events.loc[accuracy_events['exit_side'].eq(side)]
        valid_event = event_part.dropna(subset=['exit_aligned_h1_bp'])
        dashboard_rows.append({
            'side': side,
            'signal_days': int(len(event_part)),
            'H1_accuracy_pct': float(valid_event['exit_aligned_h1_bp'].gt(0).mean() * 100) if len(valid_event) else np.nan,
            'segment_win_original_pct': float(original['segment_win_rate_pct']),
            'segment_win_adjusted_pct': float(adjusted['segment_win_rate_pct']),
            'aligned_segment_mean_original_pct': float(original['mean_aligned_segment_return_pct']),
            'aligned_segment_mean_adjusted_pct': float(adjusted['mean_aligned_segment_return_pct']),
            'mean_holding_original_days': float(original['mean_holding_days']),
            'mean_holding_adjusted_days': float(adjusted['mean_holding_days']),
            'one_day_share_adjusted_pct': float(adjusted['one_day_share_pct']),
            'short_<=3d_share_adjusted_pct': float(adjusted['short_le_3d_share_pct']),
        })
    final_quality_dashboard = pd.DataFrame(dashboard_rows)
    print('最终质量仪表盘（段收益为方向对齐收益，selection_used=False）：')
    display(final_quality_dashboard.round(3))

    # 最终截图图：点形状表示退出方向，颜色表示退出后 H1 O2O 是否有利。
    curve = plot_df[['date', 'close']].dropna().sort_values('date')
    fig, ax = plt.subplots(figsize=(24, 9), dpi=120)
    ax.plot(curve['date'], curve['close'], color='#444444', linewidth=1.0, alpha=0.88, label='CSI500 close', zorder=1)
    accuracy_events = accuracy_events.sort_values('date').copy()
    accuracy_events['plot_color'] = np.select(
        [accuracy_events['exit_aligned_h1_bp'].gt(0), accuracy_events['exit_aligned_h1_bp'].lt(0)],
        ['#2ca02c', '#d62728'],
        default='#7f7f7f',
    )
    marker_map = {'下跌退出（-1→0）': 'v', '上涨退出（1→0）': '^'}
    for side in side_order:
        part = accuracy_events.loc[accuracy_events['exit_side'].eq(side)].dropna(subset=['close']).copy()
        ax.scatter(
            part['date'], part['close'], s=34, c=part['plot_color'].tolist(),
            marker=marker_map[side], edgecolors='black', linewidths=.25, alpha=.92,
            label=side, zorder=3,
        )
        for _, row in part.iterrows():
            if pd.isna(row['o2o_h1_bp']):
                continue
            offset = 8 if marker_map[side] == '^' else -8
            va = 'bottom' if offset > 0 else 'top'
            ax.annotate(
                f"{row['o2o_h1_bp']:+.0f}",
                xy=(row['date'], row['close']), xytext=(0, offset),
                textcoords='offset points', fontsize=4.5, rotation=90,
                ha='center', va=va, color=row['plot_color'], alpha=.80,
            )
    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markeredgecolor='black', markersize=7, label='green: favorable / accurate'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markeredgecolor='black', markersize=7, label='red: unfavorable / incorrect'),
        Line2D([0], [0], marker='v', color='#333333', linestyle='None', markersize=7, label='minus exit -1 to 0'),
        Line2D([0], [0], marker='^', color='#333333', linestyle='None', markersize=7, label='plus exit 1 to 0'),
    ]
    ax.legend(handles=legend_handles, ncol=4, loc='upper left', framealpha=.92)
    ax.text(
        .01, .015,
        'shape = exit side; color = H1 O2O outcome; number = raw H1 O2O (bp)',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=.3', facecolor='white', alpha=.78, edgecolor='#bbbbbb'),
    )
    ax.set_title('Final frozen exit predictions on CSI500: side, accuracy, and H1 O2O')
    ax.set_xlabel('Effective trading date (t close -> next actual trading-day open)')
    ax.set_ylabel('CSI500 close')
    ax.grid(alpha=.22)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    print('预测事件明细（数字与图中标注一致；aligned 为按退出方向折算后的 H1 O2O）：')
    display(accuracy_events[[
        'date', 'period', 'exit_side', 'prediction_result',
        'o2o_h1_bp', 'exit_aligned_h1_bp', 'prediction_correct',
    ]].round({'o2o_h1_bp': 2, 'exit_aligned_h1_bp': 2}))
    print('FINAL_ACCURACY_PLOT_END')
